In [0]:
# Notebook 2: EDA and Data Quality
# UAE Food Delivery Analysis
# -----------------------------------------

from pyspark.sql.functions import (
    col, count, when, isnull, round as spark_round,
    min, max, mean, stddev, countDistinct
)

# Load from Parquet
df = spark.read.parquet(
    "/Volumes/workspace/default/food_delivery_data/uae_food_delivery_750k.parquet"
)

print(f"Total rows: {df.count():,}")
print(f"Total columns: {len(df.columns)}")
print("\nSchema:")
df.printSchema()

In [0]:
# Check for nulls across all columns
print("Null value check:")
df.select([
    count(when(isnull(c), c)).alias(c)
    for c in df.columns
]).display()

In [0]:
# Key numeric column statistics
print("Numeric column statistics:")
df.select(
    "total_price_aed",
    "delivery_distance_km",
    "delivery_duration_mins",
    "unit_price_aed",
    "quantity",
    "order_quality_risk_score",
    "restaurant_health_score"
).describe().display()

In [0]:
# Distribution of key categorical columns
for col_name in ["city", "cuisine", "order_status", "payment_method", "traffic_level", "driver_vehicle"]:
    print(f"\n{col_name.upper()} distribution:")
    df.groupBy(col_name).count().orderBy("count", ascending=False).display()

In [0]:
# Order volume by month
from pyspark.sql.functions import month, year

df.groupBy(year("order_date").alias("year"), month("order_date").alias("month")) \
  .count() \
  .orderBy("year", "month") \
  .display()


In [0]:
# Order volume by hour of day
df.groupBy("order_hour") \
  .count() \
  .orderBy("order_hour") \
  .display()

In [0]:
# Churn risk distribution
df.groupBy("churn_risk") \
  .count() \
  .orderBy("churn_risk") \
  .display()

In [0]:
# Average order value by city
from pyspark.sql.functions import avg, round as spark_round

print("Average order value by city:")
df.groupBy("city") \
  .agg(spark_round(avg("total_price_aed"), 2).alias("avg_order_value_aed")) \
  .orderBy("avg_order_value_aed", ascending=False) \
  .display()

print("Average order value by cuisine:")
df.groupBy("cuisine") \
  .agg(spark_round(avg("total_price_aed"), 2).alias("avg_order_value_aed")) \
  .orderBy("avg_order_value_aed", ascending=False) \
  .display()

In [0]:
# Cancellation rate by restaurant health score band
from pyspark.sql.functions import when, avg, round

df.withColumn("health_band", 
    when(col("restaurant_health_score") < 0.4, "Poor")
    .when(col("restaurant_health_score") < 0.7, "Average")
    .otherwise("Good")
) \
.groupBy("health_band") \
.agg(
    spark_round(avg(when(col("order_status") == "Cancelled", 1).otherwise(0)) * 100, 2).alias("cancellation_rate_pct"),
    count("*").alias("total_orders")
) \
.orderBy("cancellation_rate_pct", ascending=False) \
.display()

In [0]:
# Average order quality risk score by traffic level
from pyspark.sql.functions import avg, round, count

df.groupBy("traffic_level") \
  .agg(
      round(avg("order_quality_risk_score"), 3).alias("avg_risk_score"),
      count("*").alias("total_orders")
  ) \
  .orderBy("avg_risk_score", ascending=False) \
  .display()